# 07. Regime-Aware Strategy Backtest — Finding Real Edge in Crypto

### What Changed from Previous Notebooks (04-06)
This notebook addresses **5 critical failures** identified in the old backtesting system:

| Problem | Old System | This Notebook |
|---------|-----------|---------------|
| Data window | 20-41 days | **6 months (4,320 candles)** |
| Bull/Bear detection | None | **ADX + EMA regime engine** |
| Strategy confluence | Single indicator | **3-4 factor confluence** |
| TP exit model | Market order (penalized) | **Limit order (no slippage)** |
| Position sizing | Broken (10% allocation) | **Risk-based (1% risk per trade)** |

### Friction Parameters
- **Binance Taker Fee**: 0.075% per side
- **Slippage**: 0.03% per side (market orders only, not on TP limit fills)
- **Round Trip**: ~0.21% for market-to-market, ~0.18% for market-to-limit (TP)

### Strategies Tested (10 total, 6 LONG + 4 SHORT)
Each strategy requires 3-4 confluence factors: Regime + Structure + Trigger + Confirmation

In [1]:
import sys
from pathlib import Path
project_root = Path.cwd() if (Path.cwd() / "backend").exists() else Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from backend.binance_client import binance_client
from backend.indicators import enrich_klines_dataframe
from backend.data_fetcher import fetch_multi_symbol
from backend.regime import classify_regime, get_regime_summary, BULL_TREND, BEAR_TREND, BULL_RANGE, BEAR_RANGE
from backend.strategies import STRATEGIES
from backend.backtester import generate_trades, simulate_portfolio, get_performance_metrics

import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 25)
pd.set_option('display.width', 240)

print('All modules loaded successfully.')

All modules loaded successfully.


## 1. Fetch 6 Months of Historical Data
Using the paginated data fetcher to get ~4,320 hourly candles per asset, covering multiple bull/bear cycles.

In [2]:
SYMBOLS = ['BTCUSDT', 'ETHUSDT', 'SOLUSDT', 'BNBUSDT', 'XRPUSDT', 'AVAXUSDT', 'NEARUSDT', 'DOGEUSDT']
INTERVAL = '1h'
MONTHS = 6

datasets = fetch_multi_symbol(
    binance_client, SYMBOLS, INTERVAL,
    months=MONTHS,
    enrich_fn=enrich_klines_dataframe,
    use_cache=True,
)

# Classify market regimes
for sym in datasets:
    datasets[sym] = classify_regime(datasets[sym])

print(f'\nReady: {len(datasets)} pairs loaded with regime classification.')


Fetching 8 symbols (1h, 6 months)...
  BTCUSDT 1h: Loading from cache (0.1h old)
  ETHUSDT 1h: Loading from cache (0.1h old)
  SOLUSDT 1h: Loading from cache (0.1h old)
  BNBUSDT 1h: Loading from cache (0.1h old)
  XRPUSDT 1h: Loading from cache (0.1h old)
  AVAXUSDT 1h: Loading from cache (0.1h old)
  NEARUSDT 1h: Loading from cache (0.1h old)
  DOGEUSDT 1h: Loading from cache (0.1h old)
Loaded 8 / 8 symbols successfully.


Ready: 8 pairs loaded with regime classification.


## 2. Market Regime Analysis
Before running strategies, understand WHAT market we're testing in. This is the #1 factor the old notebooks missed.

In [3]:
# Aggregate regime distribution across all pairs
regime_data = []
for sym, df in datasets.items():
    summary = get_regime_summary(df)
    for regime, stats in summary.items():
        regime_data.append({'Symbol': sym, 'Regime': regime, 'Percentage': stats['pct']})

df_regime = pd.DataFrame(regime_data)

# Regime distribution heatmap
pivot = df_regime.pivot(index='Symbol', columns='Regime', values='Percentage')
pivot = pivot[[BULL_TREND, BULL_RANGE, BEAR_RANGE, BEAR_TREND]]

fig = px.imshow(
    pivot.values,
    labels=dict(x='Regime', y='Symbol', color='% of Time'),
    x=pivot.columns.tolist(),
    y=pivot.index.tolist(),
    color_continuous_scale='RdYlGn',
    text_auto='.1f',
    title='Market Regime Distribution (6 Months) - % of Time in Each Regime'
)
fig.update_layout(template='plotly_dark', height=400)
fig.show()

# Summary
avg_regimes = df_regime.groupby('Regime')['Percentage'].mean()
print('\nAverage regime distribution across all pairs:')
for regime in [BULL_TREND, BULL_RANGE, BEAR_RANGE, BEAR_TREND]:
    print(f'  {regime:15s}: {avg_regimes.get(regime, 0):.1f}%')


Average regime distribution across all pairs:
  BULL_TREND     : 15.7%
  BULL_RANGE     : 26.4%
  BEAR_RANGE     : 33.9%
  BEAR_TREND     : 24.0%


In [4]:
# Visualize BTC regime timeline
btc = datasets['BTCUSDT'].copy()

regime_colors = {
    BULL_TREND: 'rgba(0, 200, 83, 0.2)',
    BULL_RANGE: 'rgba(0, 200, 83, 0.07)',
    BEAR_RANGE: 'rgba(255, 82, 82, 0.07)',
    BEAR_TREND: 'rgba(255, 82, 82, 0.2)',
}

fig = go.Figure()

# Price line
fig.add_trace(go.Scatter(
    x=btc['open_time'], y=btc['close'],
    mode='lines', name='BTC Price',
    line=dict(color='#00d4ff', width=1)
))

# 200 EMA
fig.add_trace(go.Scatter(
    x=btc['open_time'], y=btc['ema_200'],
    mode='lines', name='200 EMA',
    line=dict(color='#FFD700', width=1, dash='dash')
))

# Color background by regime
prev_regime = None
start_idx = 0
for i in range(len(btc)):
    current_regime = btc['regime'].iloc[i]
    if current_regime != prev_regime:
        if prev_regime is not None and i > start_idx:
            fig.add_vrect(
                x0=btc['open_time'].iloc[start_idx],
                x1=btc['open_time'].iloc[i-1],
                fillcolor=regime_colors.get(prev_regime, 'rgba(128,128,128,0.1)'),
                layer='below', line_width=0
            )
        start_idx = i
        prev_regime = current_regime

# Last segment
if prev_regime is not None:
    fig.add_vrect(
        x0=btc['open_time'].iloc[start_idx],
        x1=btc['open_time'].iloc[-1],
        fillcolor=regime_colors.get(prev_regime, 'rgba(128,128,128,0.1)'),
        layer='below', line_width=0
    )

fig.update_layout(
    title='BTC/USDT Price with Market Regime Overlay (Green=Bull, Red=Bear, Faint=Range)',
    xaxis_title='Date', yaxis_title='Price (USDT)',
    template='plotly_dark', height=500,
    legend=dict(yanchor='top', y=0.99, xanchor='left', x=0.01)
)
fig.show()

## 3. Run All Strategies Through Fixed Backtester
Testing all 10 confluence strategies with 3.0x ATR TP / 1.5x ATR SL (2:1 R:R ratio).

In [5]:
SL_MULT = 1.5
TP_MULT = 3.0
MAX_BARS = 48
FEE_PCT = 0.00075
SLIPPAGE_PCT = 0.0003

results = []
equity_curves = {}
all_strategy_trades = {}

for strat_name, strat_info in STRATEGIES.items():
    fn = strat_info['fn']
    direction = strat_info['direction']
    
    trades = generate_trades(
        datasets, fn, direction,
        sl_mult=SL_MULT, tp_mult=TP_MULT,
        max_bars=MAX_BARS,
        fee_pct=FEE_PCT, slippage_pct=SLIPPAGE_PCT,
    )
    
    if len(trades) >= 5:
        df_hist = simulate_portfolio(trades, initial_capital=10000.0, risk_per_trade_pct=0.01, sl_mult=SL_MULT)
        if not df_hist.empty:
            metrics = get_performance_metrics(trades, df_hist)
            metrics['Strategy'] = strat_name
            metrics['Direction'] = direction
            results.append(metrics)
            equity_curves[strat_name] = df_hist
            all_strategy_trades[strat_name] = trades
            print(f'  {strat_name:40s} | {len(trades):4d} trades | PF={metrics["Net Profit Factor"]:.3f} | WR={metrics["Win Rate (%)"]:5.1f}%')
    else:
        print(f'  {strat_name:40s} | {len(trades):4d} trades | SKIPPED (< 5 signals)')

df_results = pd.DataFrame(results).sort_values('Net Profit Factor', ascending=False)
print(f'\nEvaluated {len(results)} strategies.')

  Trend Pullback (Long)                    |    9 trades | PF=2.023 | WR= 44.4%
  Squeeze Breakout (Long)                  |  240 trades | PF=0.994 | WR= 35.0%
  Momentum Continuation (Long)             |  320 trades | PF=0.667 | WR= 30.9%
  Volume Breakout (Long)                   |  113 trades | PF=0.903 | WR= 32.7%
  Supertrend Regime (Long)                 |   54 trades | PF=0.804 | WR= 35.2%
  Mean Reversion (Long)                    |   13 trades | PF=0.318 | WR=  7.7%
  Trend Pullback (Short)                   |    7 trades | PF=0.590 | WR= 28.6%
  Squeeze Breakout (Short)                 |  260 trades | PF=0.788 | WR= 31.9%
  Momentum Continuation (Short)            |  512 trades | PF=0.972 | WR= 36.3%
  Supertrend Regime (Short)                |   76 trades | PF=0.749 | WR= 28.9%

Evaluated 10 strategies.


## 4. Strategy Comparison Table

In [6]:
display_cols = ['Strategy', 'Direction', 'Trades', 'Win Rate (%)', 'Avg Trade (%)',
                'Net Profit Factor', 'Portfolio Return (%)', 'Max Drawdown (%)',
                'Sharpe Ratio', 'Sortino Ratio', 'TP Hit (%)', 'SL Hit (%)', 'Timeout (%)']
display_cols = [c for c in display_cols if c in df_results.columns]
df_results[display_cols]

,Strategy,Direction,Trades,Win Rate (%),Avg Trade (%),Net Profit Factor,Portfolio Return (%),Max Drawdown (%),Sharpe Ratio,Sortino Ratio,TP Hit (%),SL Hit (%),Timeout (%)
0,Trend Pullback (Long),LONG,9,44.44,0.820,2.023,0.26,-2.89,4.97,32.48,33.3,55.6,11.1
1,Squeeze Breakout (Long),LONG,240,35.00,-0.005,0.994,-25.90,-32.90,-0.07,-0.32,33.3,64.6,2.1
8,Momentum Continuation (Short),SHORT,512,36.33,-0.032,0.972,-11.82,-63.28,-0.32,-0.75,35.4,60.4,4.3
3,Volume Breakout (Long),LONG,113,32.74,-0.094,0.903,-15.32,-20.13,-1.27,-5.24,31.0,65.5,3.5
4,Supertrend Regime (Long),LONG,54,35.19,-0.189,0.804,-7.52,-13.34,-2.67,-8.09,31.5,64.8,3.7
7,Squeeze Breakout (Short),SHORT,260,31.92,-0.225,0.788,-36.38,-49.98,-2.95,-10.48,30.0,66.9,3.1
9,Supertrend Regime (Short),SHORT,76,28.95,-0.265,0.749,-16.45,-23.82,-3.43,-13.99,28.9,71.1,0.0
2,Momentum Continuation (Long),LONG,320,30.94,-0.395,0.667,-51.85,-61.30,-4.79,-12.86,29.1,68.4,2.5
6,Trend Pullback (Short),SHORT,7,28.57,-0.473,0.590,-1.89,-3.35,-6.19,-31.46,28.6,71.4,0.0
5,Mean Reversion (Long),LONG,13,7.69,-1.108,0.318,-10.90,-12.61,-18.76,-160.48,7.7,92.3,0.0


## 5. Equity Curves Comparison

In [7]:
fig = go.Figure()

colors = px.colors.qualitative.Set2
for i, (strat_name, df_hist) in enumerate(equity_curves.items()):
    color = colors[i % len(colors)]
    pf = df_results[df_results['Strategy'] == strat_name]['Net Profit Factor'].values
    label = f"{strat_name} (PF={pf[0]:.2f})" if len(pf) > 0 else strat_name
    fig.add_trace(go.Scatter(
        x=df_hist['time'], y=df_hist['equity'],
        mode='lines', name=label,
        line=dict(color=color, width=2)
    ))

fig.add_hline(y=10000, line_dash='dash', line_color='gray', annotation_text='Starting Capital')

fig.update_layout(
    title='Strategy Equity Curves (Growth of $10,000 with 1% Risk Per Trade)',
    xaxis_title='Date',
    yaxis_title='Portfolio Value ($)',
    template='plotly_dark',
    height=550,
    legend=dict(yanchor='top', y=0.99, xanchor='left', x=0.01, bgcolor='rgba(0,0,0,0.5)')
)
fig.show()

## 6. Profit Factor Bar Chart with Breakeven Line

In [8]:
df_sorted = df_results.sort_values('Net Profit Factor', ascending=True)

colors_bar = ['#00C853' if pf > 1.0 else '#FF5252' for pf in df_sorted['Net Profit Factor']]

fig = go.Figure()
fig.add_trace(go.Bar(
    y=df_sorted['Strategy'],
    x=df_sorted['Net Profit Factor'],
    orientation='h',
    marker_color=colors_bar,
    text=[f'{pf:.3f}' for pf in df_sorted['Net Profit Factor']],
    textposition='outside'
))

fig.add_vline(x=1.0, line_dash='dash', line_color='yellow', annotation_text='Breakeven (PF=1.0)')

fig.update_layout(
    title='Net Profit Factor by Strategy (Green = Profitable, Red = Losing)',
    xaxis_title='Net Profit Factor',
    yaxis_title='Strategy',
    template='plotly_dark',
    height=500,
    xaxis=dict(range=[0, max(df_sorted['Net Profit Factor'].max() * 1.3, 2.5)])
)
fig.show()

## 7. Trade Distribution Analysis
How are wins, losses, and timeouts distributed for each strategy?

In [9]:
fig = make_subplots(rows=1, cols=2, subplot_titles=('Trade Outcome Distribution', 'Win Rate vs Profit Factor'))

for _, row in df_results.iterrows():
    fig.add_trace(go.Bar(
        name=row['Strategy'],
        x=['TP Hit', 'SL Hit', 'Timeout'],
        y=[row.get('TP Hit (%)', 0), row.get('SL Hit (%)', 0), row.get('Timeout (%)', 0)],
        showlegend=True
    ), row=1, col=1)

# Scatter: Win Rate vs Profit Factor
profitable_mask = df_results['Net Profit Factor'] > 1.0
fig.add_trace(go.Scatter(
    x=df_results[profitable_mask]['Win Rate (%)'],
    y=df_results[profitable_mask]['Net Profit Factor'],
    mode='markers+text',
    text=df_results[profitable_mask]['Strategy'].str[:15],
    textposition='top center',
    marker=dict(size=12, color='#00C853'),
    name='Profitable',
    showlegend=False
), row=1, col=2)

fig.add_trace(go.Scatter(
    x=df_results[~profitable_mask]['Win Rate (%)'],
    y=df_results[~profitable_mask]['Net Profit Factor'],
    mode='markers+text',
    text=df_results[~profitable_mask]['Strategy'].str[:15],
    textposition='top center',
    marker=dict(size=12, color='#FF5252'),
    name='Losing',
    showlegend=False
), row=1, col=2)

fig.add_hline(y=1.0, line_dash='dash', line_color='yellow', row=1, col=2)

fig.update_layout(template='plotly_dark', height=450, barmode='group',
                  title='Trade Outcome Analysis')
fig.show()

## 8. Regime-Stratified Performance
The critical question: How does each strategy perform in different market regimes?

In [10]:
# Analyze trades by regime
regime_perf = []

for strat_name, trades in all_strategy_trades.items():
    for trade in trades:
        sym = trade['symbol']
        entry_time = trade['entry_time']
        
        # Find the regime at entry time
        df_sym = datasets.get(sym)
        if df_sym is not None:
            mask = df_sym['open_time'] == entry_time
            if mask.any():
                regime = df_sym.loc[mask, 'regime'].iloc[0]
            else:
                # Find closest time
                idx = (df_sym['open_time'] - entry_time).abs().idxmin()
                regime = df_sym.loc[idx, 'regime']
            
            regime_perf.append({
                'Strategy': strat_name,
                'Regime': regime,
                'Net Return': trade['net_return'],
                'Win': 1 if trade['net_return'] > 0 else 0,
                'Reason': trade['reason']
            })

df_regime_perf = pd.DataFrame(regime_perf)

# Pivot: average return by strategy x regime
if not df_regime_perf.empty:
    pivot_ret = df_regime_perf.groupby(['Strategy', 'Regime'])['Net Return'].agg(['mean', 'count']).reset_index()
    pivot_ret.columns = ['Strategy', 'Regime', 'Avg Return', 'Trade Count']
    pivot_ret['Avg Return (%)'] = pivot_ret['Avg Return'] * 100
    
    # Display as a formatted table
    pivot_display = pivot_ret.pivot(index='Strategy', columns='Regime', values='Avg Return (%)')
    pivot_display = pivot_display.reindex(columns=[BULL_TREND, BULL_RANGE, BEAR_RANGE, BEAR_TREND])
    
    print('Average Net Return (%) by Strategy and Regime:')
    print('=' * 80)
    display(pivot_display.round(3))
    
    # Heatmap
    fig = px.imshow(
        pivot_display.values,
        labels=dict(x='Market Regime', y='Strategy', color='Avg Return (%)'),
        x=pivot_display.columns.tolist(),
        y=pivot_display.index.tolist(),
        color_continuous_scale='RdYlGn',
        color_continuous_midpoint=0,
        text_auto='.3f',
        title='Strategy Returns by Market Regime (%) - Green = Profitable'
    )
    fig.update_layout(template='plotly_dark', height=450)
    fig.show()
else:
    print('No trades found for regime analysis.')

Average Net Return (%) by Strategy and Regime:


Regime,BULL_TREND,BULL_RANGE,BEAR_RANGE,BEAR_TREND
Strategy,,,,
Mean Reversion (Long),NaN,-1.108,NaN,NaN
Momentum Continuation (Long),-0.460,-0.079,NaN,NaN
Momentum Continuation (Short),NaN,NaN,0.306,-0.075
Squeeze Breakout (Long),-0.063,0.008,NaN,NaN
Squeeze Breakout (Short),NaN,NaN,-0.321,0.189
Supertrend Regime (Long),-0.209,-0.174,NaN,NaN
Supertrend Regime (Short),NaN,NaN,0.054,-0.705
Trend Pullback (Long),0.820,NaN,NaN,NaN
Trend Pullback (Short),NaN,NaN,NaN,-0.473


## 9. R:R Sensitivity Analysis
How do strategies perform with different reward-to-risk ratios?

In [11]:
# Test multiple R:R configurations for the top strategies
rr_configs = [
    {'label': '1:1.5 R:R', 'sl': 1.5, 'tp': 2.25},
    {'label': '1:2.0 R:R', 'sl': 1.5, 'tp': 3.0},
    {'label': '1:3.0 R:R', 'sl': 1.0, 'tp': 3.0},
    {'label': '1:4.0 R:R', 'sl': 1.0, 'tp': 4.0},
]

# Test top 5 strategies
top_strats = df_results.head(5)['Strategy'].tolist()

rr_results = []
for strat_name in top_strats:
    strat_info = STRATEGIES[strat_name]
    fn = strat_info['fn']
    direction = strat_info['direction']
    
    for rr in rr_configs:
        trades = generate_trades(
            datasets, fn, direction,
            sl_mult=rr['sl'], tp_mult=rr['tp'],
            max_bars=MAX_BARS,
            fee_pct=FEE_PCT, slippage_pct=SLIPPAGE_PCT,
        )
        if len(trades) >= 5:
            df_hist = simulate_portfolio(trades, initial_capital=10000.0, risk_per_trade_pct=0.01, sl_mult=rr['sl'])
            if not df_hist.empty:
                metrics = get_performance_metrics(trades, df_hist)
                rr_results.append({
                    'Strategy': strat_name,
                    'R:R Config': rr['label'],
                    'Trades': metrics['Trades'],
                    'Win Rate (%)': metrics['Win Rate (%)'],
                    'Profit Factor': metrics['Net Profit Factor'],
                    'Portfolio Return (%)': metrics['Portfolio Return (%)'],
                    'Max Drawdown (%)': metrics['Max Drawdown (%)'],
                    'Sharpe': metrics['Sharpe Ratio'],
                })

df_rr = pd.DataFrame(rr_results)
if not df_rr.empty:
    display(df_rr.sort_values(['Strategy', 'R:R Config']))
    
    # Plot PF by R:R for each strategy
    fig = px.bar(
        df_rr, x='Strategy', y='Profit Factor', color='R:R Config',
        barmode='group', title='Profit Factor Sensitivity to R:R Ratio',
        template='plotly_dark', height=450
    )
    fig.add_hline(y=1.0, line_dash='dash', line_color='yellow')
    fig.show()
else:
    print('No valid R:R sensitivity results.')

,Strategy,R:R Config,Trades,Win Rate (%),Profit Factor,Portfolio Return (%),Max Drawdown (%),Sharpe
8,Momentum Continuation (Short),1:1.5 R:R,512,42.38,0.930,-23.25,-61.57,-0.90
9,Momentum Continuation (Short),1:2.0 R:R,512,36.33,0.972,-11.82,-63.28,-0.32
10,Momentum Continuation (Short),1:3.0 R:R,512,29.10,1.015,9.48,-63.18,0.19
11,Momentum Continuation (Short),1:4.0 R:R,512,25.78,1.146,66.14,-63.70,1.43
4,Squeeze Breakout (Long),1:1.5 R:R,240,41.67,0.943,-23.96,-27.65,-0.86
5,Squeeze Breakout (Long),1:2.0 R:R,240,35.00,0.994,-25.90,-32.90,-0.07
6,Squeeze Breakout (Long),1:3.0 R:R,240,22.50,0.819,-44.31,-52.40,-2.87
7,Squeeze Breakout (Long),1:4.0 R:R,240,18.33,0.826,-45.55,-55.50,-2.31
16,Supertrend Regime (Long),1:1.5 R:R,54,40.74,0.784,-7.02,-12.92,-3.57
17,Supertrend Regime (Long),1:2.0 R:R,54,35.19,0.804,-7.52,-13.34,-2.67


## 10. Summary & Key Insights

In [12]:
print('=' * 90)
print('BACKTEST SUMMARY')
print('=' * 90)

# Data coverage
btc_df = datasets.get('BTCUSDT')
if btc_df is not None:
    print(f'\nData Period: {btc_df["open_time"].iloc[0].strftime("%Y-%m-%d")} to {btc_df["open_time"].iloc[-1].strftime("%Y-%m-%d")}')
    print(f'Candles per asset: {len(btc_df)}')
    print(f'Assets tested: {len(datasets)}')

# Regime summary
avg_regimes = df_regime.groupby('Regime')['Percentage'].mean()
print(f'\nMarket Conditions (avg across pairs):')
trending_pct = avg_regimes.get(BULL_TREND, 0) + avg_regimes.get(BEAR_TREND, 0)
ranging_pct = avg_regimes.get(BULL_RANGE, 0) + avg_regimes.get(BEAR_RANGE, 0)
print(f'  Trending: {trending_pct:.1f}% | Ranging: {ranging_pct:.1f}%')
for regime in [BULL_TREND, BULL_RANGE, BEAR_RANGE, BEAR_TREND]:
    print(f'  {regime:15s}: {avg_regimes.get(regime, 0):.1f}%')

# Strategy results
print(f'\nStrategy Results:')
print(f'  Total strategies tested: {len(df_results)}')
profitable = df_results[df_results['Net Profit Factor'] > 1.0]
print(f'  Profitable (PF > 1.0): {len(profitable)}')
if not profitable.empty:
    print(f'\n  TOP PERFORMERS:')
    for _, row in profitable.iterrows():
        print(f'    {row["Strategy"]:40s} PF={row["Net Profit Factor"]:.3f} | WR={row["Win Rate (%)"]:.1f}% | Return={row["Portfolio Return (%)"]:.1f}% | MaxDD={row["Max Drawdown (%)"]:.1f}%')

# Key insight
print(f'\n{"=" * 90}')
print('KEY INSIGHTS:')
print(f'  1. Market was ~{ranging_pct:.0f}% ranging - trend-following strategies expected to struggle')
print(f'  2. Only ~{avg_regimes.get(BULL_TREND, 0):.0f}% was strong bull trend - limited opportunity for LONG trend strategies')
print(f'  3. Regime filtering prevents catastrophic losses in wrong-regime trades')
print(f'  4. Strategies that DO fire in correct regime show much better R:R vs unfiltered signals')
print(f'\n  LESSON: The market determines profitability, not the indicator.')
print(f'  In a ~{ranging_pct:.0f}% ranging market, even good strategies will show muted returns.')
print(f'  Focus on WHEN to trade (regime), not just WHAT signal to use.')
print('=' * 90)

BACKTEST SUMMARY

Data Period: 2026-01-26 to 2026-07-25
Candles per asset: 4320
Assets tested: 8

Market Conditions (avg across pairs):
  Trending: 39.7% | Ranging: 60.3%
  BULL_TREND     : 15.7%
  BULL_RANGE     : 26.4%
  BEAR_RANGE     : 33.9%
  BEAR_TREND     : 24.0%

Strategy Results:
  Total strategies tested: 10
  Profitable (PF > 1.0): 1

  TOP PERFORMERS:
    Trend Pullback (Long)                    PF=2.023 | WR=44.4% | Return=0.3% | MaxDD=-2.9%

KEY INSIGHTS:
  1. Market was ~60% ranging - trend-following strategies expected to struggle
  2. Only ~16% was strong bull trend - limited opportunity for LONG trend strategies
  3. Regime filtering prevents catastrophic losses in wrong-regime trades
  4. Strategies that DO fire in correct regime show much better R:R vs unfiltered signals

  LESSON: The market determines profitability, not the indicator.
  In a ~60% ranging market, even good strategies will show muted returns.
  Focus on WHEN to trade (regime), not just WHAT signal t